<a href="https://colab.research.google.com/github/rana-17/TradingAgentCode-/blob/main/TradingAgentCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-groq

In [ ]:
!pip install langchain-google-genai

In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

google_api_key = userdata.get('googleapikey')
model = init_chat_model(
"google_genai:gemini-2.5-flash",
api_key=google_api_key
)

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[],
    system_prompt="You are a stock trading analyst."
)

In [ ]:
import requests
from google.colab import userdata

twelvedata_api_key = userdata.get('twelveapi')

url = "https://api.twelvedata.com/price"
params = {
    "symbol": "AAPL",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)

In [ ]:
from langchain.tools import tool

@tool
def get_stock_price(ticker: str) -> str:
    """Get the current market price for a stock. Example: ticker='AAPL'"""
    url = "https://api.twelvedata.com/price"
    params = {"symbol": ticker, "apikey": twelvedata_api_key}
    response = requests.get(url, params=params)
    data = response.json()
    if "price" not in data:
        return f"Could not fetch price for {ticker}"
    return f"{ticker} current price: ${float(data['price']):.2f}"

In [ ]:
print(get_stock_price.invoke({"ticker": "AAPL"}))

In [ ]:
url = "https://api.twelvedata.com/time_series"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "outputsize": "5",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)

In [ ]:
@tool
def get_price_history(ticker: str) -> str:
    """Get the last 10 days of closing prices for a stock. Example: ticker='AAPL'"""
    url = "https://api.twelvedata.com/time_series"
    params = {
        "symbol": ticker,
        "interval": "1day",
        "outputsize": "10",
        "apikey": twelvedata_api_key
    }
    response = requests.get(url, params=params)
    data = response.json()
    if "values" not in data:
        return f"Could not fetch price history for {ticker}"
    prices = [
        f"  {item['datetime']}: ${float(item['close']):.2f}"
        for item in reversed(data["values"])
    ]
    return f"{ticker} closing prices (last 10 days):\n" + "\n".join(prices)

In [ ]:
print(get_price_history.invoke({"ticker": "NVDA"}))

In [ ]:
url = "https://api.twelvedata.com/rsi"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "time_period": "14",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)

In [ ]:
@tool
def calculate_rsi(ticker: str) -> str:
    """Calculate the current RSI for a stock to check if it is overbought or oversold. Example: ticker='AAPL'"""
    url = "https://api.twelvedata.com/rsi"
    params = {
        "symbol": ticker,
        "interval": "1day",
        "time_period": "14",
        "apikey": twelvedata_api_key
    }
    response = requests.get(url, params=params)
    data = response.json()
    if "values" not in data:
        return f"Could not fetch RSI for {ticker}"
    rsi = float(data["values"][0]["rsi"])
    if rsi > 70:
        signal = "Overbought — possible pullback ahead"
    elif rsi < 30:
        signal = "Oversold — possible recovery ahead"
    else:
        signal = "Neutral"
    return f"{ticker} RSI(14): {rsi:.1f} → {signal}"

In [ ]:
print(calculate_rsi.invoke({"ticker": "AAPL"}))

In [ ]:
url = "https://api.twelvedata.com/macd"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)

In [ ]:
url = "https://api.twelvedata.com/sma"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "time_period": "50",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)

In [ ]:
@tool
def get_sma(ticker: str) -> str:
    """Get the 50-day and 200-day Simple Moving Averages to check the long-term trend. Example: ticker='AAPL'"""
    base_params = {"symbol": ticker, "interval": "1day", "apikey": twelvedata_api_key}

    response_50 = requests.get("https://api.twelvedata.com/sma", params={**base_params, "time_period": "50"})
    response_200 = requests.get("https://api.twelvedata.com/sma", params={**base_params, "time_period": "200"})

    data_50 = response_50.json()
    data_200 = response_200.json()

    if "values" not in data_50 or "values" not in data_200:
        return f"Could not fetch SMA for {ticker}"

    sma_50 = float(data_50["values"][0]["sma"])
    sma_200 = float(data_200["values"][0]["sma"])

    if sma_50 > sma_200:
        cross = "Golden Cross — Bullish long-term signal"
    else:
        cross = "Death Cross — Bearish long-term signal"

    return f"{ticker} 50-day SMA: ${sma_50:.2f} | 200-day SMA: ${sma_200:.2f} | {cross}"

In [ ]:
print(get_sma.invoke({"ticker": "MSFT"}))

In [ ]:
system_prompt = """You are a stock trading analyst.

When asked to analyse a stock:
1. Get the current price using get_stock_price
2. Get the recent price history using get_price_history
3. Calculate RSI using calculate_rsi
4. Calculate MACD using calculate_macd
5. Get the SMA using get_sma
6. Combine all the data and provide:
   - DECISION: BUY / SELL / HOLD
   - CONFIDENCE: High / Medium / Low
   - A short rationale based on the signals

Do not skip any tool. Always run all five before giving a decision."""

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[get_stock_price, get_price_history, calculate_rsi, get_sma],
    system_prompt=system_prompt
)

In [ ]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "Should I buy Apple stock?"}]
})
print(response["messages"][-1].content)

In [ ]:

response = agent.invoke({
    "messages": [{"role": "user", "content": "Apple vs Microsoft — which looks technically stronger right now?"}]
})
print(response["messages"][-1].content)

In [ ]:
!pip install finnhub-python

In [ ]:
import finnhub
from google.colab import userdata

finnhub_client = finnhub.Client(api_key=userdata.get('finnhubapikey'))

In [ ]:
from datetime import datetime, timedelta

today = datetime.now().strftime("%Y-%m-%d")
week_ago = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")

news = finnhub_client.company_news("AAPL", _from=week_ago, to=today)
print(news[:2])

In [ ]:
from langchain.tools import tool
from datetime import datetime, timedelta

@tool
def get_company_news(ticker: str) -> str:
    """Get the latest 5 news headlines for a stock from the past 7 days. Example: ticker='AAPL'"""
    today = datetime.now().strftime("%Y-%m-%d")
    week_ago = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")
    news = finnhub_client.company_news(ticker, _from=week_ago, to=today)
    if not news:
        return f"No recent news found for {ticker} in the past 7 days."
    headlines = [f"  • {item['headline']}" for item in news[:5]]
    return f"Latest news for {ticker}:\n" + "\n".join(headlines)

In [ ]:
import requests
from google.colab import userdata

ALPACA_BASE_URL = "https://paper-api.alpaca.markets"

alpaca_headers = {
    "APCA-API-KEY-ID": userdata.get("alpacaapi"),
    "APCA-API-SECRET-KEY": userdata.get("alpacasecret")
}

response = requests.get(f"{ALPACA_BASE_URL}/v2/account", headers=alpaca_headers)

In [ ]:
response = requests.post(
    f"{ALPACA_BASE_URL}/v2/orders",
    json={
        "symbol": "AAPL",
        "qty": "1",
        "side": "buy",
        "type": "market",
        "time_in_force": "day"
    },
    headers={**alpaca_headers, "Content-Type": "application/json"}
)

In [ ]:
@tool
def place_order(ticker: str, side: str, quantity: int) -> str:
    """Place a paper trade market order on Alpaca. side must be 'buy' or 'sell'. Example: ticker='AAPL', side='buy', quantity=1"""
    response = requests.post(
        f"{ALPACA_BASE_URL}/v2/orders",
        json={
            "symbol": ticker,
            "qty": str(quantity),
            "side": side.lower(),
            "type": "market",
            "time_in_force": "day"
        },
        headers={**alpaca_headers, "Content-Type": "application/json"}
    )
    data = response.json()
    if "id" not in data:
        return f"Order failed: {data.get('message', 'Unknown error')}"
    return f"Paper trade placed: {side.upper()} {quantity} share(s) of {ticker} | Order ID: {data['id']}"

In [ ]:
response = requests.get(f"{ALPACA_BASE_URL}/v2/positions", headers=alpaca_headers)

In [ ]:
@tool
def get_stocks_owned() -> str:
    """Check what stocks we currently own and if we are making profit or loss."""
    response = requests.get(f"{ALPACA_BASE_URL}/v2/positions", headers=alpaca_headers)
    positions = response.json()
    if not positions:
        return "You do not own any stocks yet."
    result = "Stocks you own:\n"
    for p in positions:
        result += (
            f"  {p['symbol']}: {p['qty']} share(s) | "
            f"Bought at: ${float(p['avg_entry_price']):.2f} | "
            f"Profit/Loss: ${float(p['unrealized_pl']):.2f}\n"
        )
    return result

In [ ]:
@tool
def get_balance() -> str:
    """Check how much virtual money we have left to buy stocks."""
    response = requests.get(f"{ALPACA_BASE_URL}/v2/account", headers=alpaca_headers)
    data = response.json()
    return (
        f"Total account value: ${float(data['equity']):,.2f} | "
        f"Available to spend: ${float(data['buying_power']):,.2f}"
    )

In [ ]:
print(get_balance.invoke({}))
# Total account value: $100,000.00 | Available to spend: $200,000.00

print(get_stocks_owned.invoke({}))
# You do not own any stocks yet.

In [ ]:
system_prompt = """You are a professional stock trading analyst with the ability to place paper trades.

When asked to analyse a stock, follow these steps in order:
1. Get the current price → get_stock_price
2. Get the recent price history → get_price_history
3. Calculate RSI → calculate_rsi
4. Get the 50-day and 200-day SMA → get_sma
5. Get recent news headlines → get_company_news
6. Read the headlines and judge whether the news is Positive, Neutral, or Negative for the stock
7. Look at all the data together and output the analysis in this format:

═══════════════════════════════════════════
STOCK ANALYSIS REPORT: [TICKER]
═══════════════════════════════════════════

DECISION: BUY / SELL / HOLD
CONFIDENCE: High / Medium / Low

Current Price: $[price]
Recent trend: [describe direction from price history]

RSI(14): [value] → [Overbought / Neutral / Oversold]
SMA: [Golden Cross / Death Cross] — [short explanation]

News: [Positive / Neutral / Negative]
Key headlines:
  • [headline 1]
  • [headline 2]

Rationale: [2-3 sentences combining all signals to explain the decision]

Risk Note: This is a paper trading simulation for educational purposes only.
═══════════════════════════════════════════

EXECUTION RULE:
- DECISION is BUY with High or Medium confidence → call place_order to buy 1 share
- DECISION is SELL with High confidence and you already hold the stock → call place_order to sell 1 share
- DECISION is HOLD → do not place any order"""

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from google.colab import userdata

google_api_key = userdata.get('googleapikey')
model = init_chat_model("google_genai:gemini-2.5-flash", api_key=google_api_key)

agent = create_agent(
    model=model,
    tools=[
        get_stock_price,
        get_price_history,
        calculate_rsi,
        get_sma,
        get_company_news,
        place_order,
        get_stocks_owned,
        get_balance
    ],
    system_prompt=system_prompt
)

In [ ]:
from pprint import pprint
response = agent.invoke({
    "messages": [{"role": "user", "content": "Should I buy Nvidia today?"}]
})
pprint(response["messages"][-1].content)